Imports and config

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
from ultralytics import YOLO
import re, random, shutil, json
from collections import defaultdict, Counter
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


REPO = Path("..").resolve()
DATA = REPO / "data"

INCOMING = DATA / "incoming"
ACCEPTED = DATA / "auto_accepted"
REVIEW = DATA / "needs_review"
META = DATA / "meta"
RAW = DATA / "raw"

ACCEPTED_IMG = ACCEPTED / "images"
ACCEPTED_LBL = ACCEPTED / "labels"
REVIEW_IMG = REVIEW / "images"

for p in [INCOMING, ACCEPTED_IMG, ACCEPTED_LBL, REVIEW_IMG, META, RAW]:
    p.mkdir(parents=True, exist_ok=True)

print("Incoming:", INCOMING)
print("Accepted:", ACCEPTED)
print("Review:", REVIEW)

Split by Tray ID (for use with augmentations)

In [ ]:
POOL_IMG = DATA / "raw" / "images"
POOL_LBL = DATA / "raw" / "labels"

OUT_TRAIN = DATA / "train"
OUT_VAL   = DATA / "val_locked"
OUT_TEST  = DATA / "test_locked"
META = DATA / "meta"
META.mkdir(parents=True, exist_ok=True)

def tray_id_from_stem(stem: str):
    # matches tray_0001 etc anywhere in the name (start is ideal but this is safer)
    m = re.search(r"((?:tray|img)_\d+)", stem, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

# group images by tray_id
groups = defaultdict(list)
missing_labels = []
for img in sorted(POOL_IMG.glob("*.jpg")):
    tid = tray_id_from_stem(img.stem)
    if tid is None:
        continue
    lbl = POOL_LBL / f"{img.stem}.txt"
    if not lbl.exists():
        missing_labels.append(img.name)
        continue
    groups[tid].append(img)

tray_ids = sorted(groups.keys())
print("Unique tray IDs:", len(tray_ids))
print("Total labeled images:", sum(len(v) for v in groups.values()))
print("Missing labels:", len(missing_labels))

# split ratios (by tray_id)
seed = 42
random.seed(seed)
random.shuffle(tray_ids)

n = len(tray_ids)
n_test = max(1, int(0.10 * n))
n_val  = max(1, int(0.15 * n))
n_train = n - n_val - n_test

test_ids = set(tray_ids[:n_test])
val_ids  = set(tray_ids[n_test:n_test+n_val])
train_ids = set(tray_ids[n_test+n_val:])

print("Tray split:", {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)})

def reset_split_dir(d: Path):
    if d.exists():
        shutil.rmtree(d)
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)

for d in [OUT_TRAIN, OUT_VAL, OUT_TEST]:
    reset_split_dir(d)

def copy_ids(id_set, out_dir: Path):
    out_img = out_dir / "images"
    out_lbl = out_dir / "labels"
    c = 0
    for tid in id_set:
        for img in groups[tid]:
            lbl = POOL_LBL / f"{img.stem}.txt"
            shutil.copy2(img, out_img / img.name)
            shutil.copy2(lbl, out_lbl / lbl.name)
            c += 1
    return c

cts = {
    "train": copy_ids(train_ids, OUT_TRAIN),
    "val":   copy_ids(val_ids, OUT_VAL),
    "test":  copy_ids(test_ids, OUT_TEST),
}

manifest = {
    "seed": seed,
    "split_by": "tray_id",
    "tray_counts": {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)},
    "image_counts": cts,
}

(META / "group_split_manifest.json").write_text(json.dumps(manifest, indent=2))
print("✅ Done. Image counts:", cts)
print("Manifest:", META / "group_split_manifest.json")

Remove duplicate images from incoming

In [ ]:

RAW_IMG = DATA / 'raw' / 'images'
IN_IMG  = DATA / "incoming" 

def tray_id(stem: str):
    m = re.search(r"((?:tray|img)_\d+)", stem, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

raw_ids = set()
for p in RAW_IMG.glob("*.jpg"):
    tid = tray_id(p.stem)
    if tid:
        raw_ids.add(tid)

to_remove = []
for p in IN_IMG.glob("*.jpg"):
    tid = tray_id(p.stem)
    if tid and tid in raw_ids:
        to_remove.append(p)

print("Incoming images:", len(list(IN_IMG.glob('*.jpg'))))
print("Will remove duplicates:", len(to_remove))

# move them to a quarantine folder instead of deleting
QUAR = DATA / "incoming_duplicates"
QUAR.mkdir(parents=True, exist_ok=True)

for p in to_remove:
    shutil.move(str(p), str(QUAR / p.name))

print("✅ Moved duplicates to:", QUAR)

Train Model

In [ ]:
DATASET_YAML = str((REPO / "data/tray_seg_data.yaml").resolve())

# pick a base model (seg)
BASE = "yolov8n-seg.pt"   # n is fast; use s if you want a bit more capacity
model = YOLO(BASE)

results = model.train(
data=DATASET_YAML,
    epochs=150,
    imgsz=640,
    batch=8,
    device="cuda",
    retina_masks=True,
    # more augmentation to compensate for small dataset
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    fliplr=0.5,
    flipud=0.1,        # trays are usually portrait but a little helps
    mosaic=1.0,
    copy_paste=0.3,    # really useful for segmentation with small datasets
    name="trayseg_v18"
)

Phase 2 Fine-tuning at imgsz=1024

Fine-tune the Phase 1 best weights at higher resolution so the mask grid is 256×256 instead of 160×160, giving much cleaner polygon geometry on high-res images.

Tips:
- Batch is halved vs Phase 1 to stay within VRAM at 1024.
- Shorter run (50 epochs) with a lower starting LR so we refine rather than overwrite Phase 1 learning.
- `mosaic=0.5` (reduced) because at 1024 mosaic tiles are ~512px — fine, but worth dialing back a bit so the model still sees full trays often.
- `copy_paste=0.3` kept — still very useful for segmentation.

In [ ]:
DATASET_YAML = str((REPO / "data/tray_seg_data.yaml").resolve())
# ---- Phase 2: fine-tune at 1024 ----
# Point at the best weights from Phase 1 (CHANGE ME FOR NEW MODELS)
PHASE1_WEIGHTS = REPO / "notebooks" / "runs" / "segment" / "trayseg_v18" / "weights" / "best.pt"

model_p2 = YOLO(str(PHASE1_WEIGHTS))

results_p2 = model_p2.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=1024,          # ← key change: 4× more mask grid area than 640
    batch=4,             # halved from Phase 1 to fit in VRAM at 1024
    device="cuda",
    retina_masks=True,
    lr0=0.001,           # lower LR — we're fine-tuning, not training from scratch
    lrf=0.01,
    warmup_epochs=1,
    # augmentation — same palette, mosaic dialled back slightly
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    fliplr=0.5,
    flipud=0.1,
    mosaic=0.5,          # reduced: at 1024 tiles are ~512px, still fine but don't overdo it
    copy_paste=0.3,
    name="trayseg_v18_1024"  # bump version so runs don't collide
)

# Copy best weights to models/ for easy access
import shutil as _sh
_best = REPO / "notebooks" / "runs" / "segment" / "trayseg_v18_1024" / "weights" / "best.pt"
_dst  = REPO / "models" / "trayseg_v18_1024.pt"
if _best.exists():
    _sh.copy2(_best, _dst)
    print(f"✅ Saved Phase 2 weights → {_dst}")
else:
    print("⚠️  best.pt not found at expected path — check the run name above")


Load Model Weights

- trayseg_v1: Model trained on small dataset (54 images) augmented (twice per image) and resized (3000x4000 -> 640x640 stretched) by Roboflow

- trayseg_v2: Model trained on larger dataset (~150 images) resized and augmented by Roboflow

- trayseg_v3: Model trained on larger dataset (~150 images) resized by Roboflow

- trayseg_v4: Model trained on manually resized dataset of 100 images labelled on Roboflow

- trayseg_v5: Model trained on 640x640 dataset of 136 images: 100 labelled by hand, 36 accepted based on high prediction confidence + manual review

- trayseg_v6 Model trained on dataset of 235 images, same 100 labelled by hand, rest bootstrapped with manual review

- trayseg_v7: Model trained on 315 images and 150 epochs (all prev used 100), same split of 100 / 215

- trayseg_v8: Trained on 368 images and 150 epochs

- trayseg_v9: Trained on 383 images and 300 epochs

- trayseg_v10: Trained on 525 images (mixed 387 640x640 from first data collection and 138 4000x5000 images from second data collection), 250 epochs (ended early) 
    - bad final scored compared to v9

- trayseg_v11: Trained on 687 images (all 640x640), 300 epochs
- trayseg_v12: Trained on 765 images "", 300 epochs
- trayseg_v13: Trained on 661 1024x1024 images labeled via manual review of v12 predictions on resized original dataset of 1036 images, 300 epochs
- trayseg_v14: Trained on 757 ""
- trayseg_v15: Trained on 100 hand labelled 3000x4000 (raw size) images over 150 epochs. Changed labels to capture inner part of tray and leave out background, with the corners of the masks corresponding with inner corners of the tray rather than outer
- trayseg_v16: Trained on 288 images (100 labeled by hand, 188 labeled w high confidence predictions of v15)
- trayseg_v17_1024: Trained on same 288 images, with a fine tuning phase at imgsz = 1024 for 100 epochs
- trayseg_v18_1024: Trained on 569 images + fine tuning with higher image resolution

In [ ]:
MODEL_PATH = REPO / "models" / "trayseg_v18_1024.pt"  
model = YOLO(str(MODEL_PATH))
print("Loaded:", MODEL_PATH)

Helper Functions

In [ ]:
def contour_area(poly: np.ndarray) -> float:
    return float(cv2.contourArea(poly.astype(np.float32)))

def poly_to_yolo_seg(poly: np.ndarray, w: int, h: int) -> str:
    p = poly.astype(np.float64).copy()
    p[:, 0] /= max(w, 1)
    p[:, 1] /= max(h, 1)
    p = np.clip(p, 0.0, 1.0)
    return " ".join([f"{x:.6f} {y:.6f}" for x, y in p])

def choose_best_instance(res):
    """Return (idx, conf). Uses conf if available, else largest area."""
    if res.masks is None or res.masks.xy is None or len(res.masks.xy) == 0:
        return None, None

    polys = res.masks.xy
    confs = None
    if res.boxes is not None and res.boxes.conf is not None and len(res.boxes.conf) == len(polys):
        confs = res.boxes.conf.detach().cpu().numpy()
        idx = int(np.argmax(confs))
        return idx, float(confs[idx])

    areas = [contour_area(p) for p in polys]
    idx = int(np.argmax(areas)) if areas else None
    return idx, None

Acceptance Criteria

In [ ]:
CLASS_ID = 0  # Outer-Tray

CONF_ACCEPT = 0.7
MIN_AREA_FRAC = 0.02
MAX_AREA_FRAC = 0.99

print("CONF_ACCEPT:", CONF_ACCEPT)
print("AREA_FRAC:", (MIN_AREA_FRAC, MAX_AREA_FRAC))

Evaluate incoming images using loaded model

In [ ]:
BATCH = 4          # start tiny; increase later
IMGSZ = 1024
PRED_CONF = 0.05   # low for detection; we gate ourselves
DEVICE = 'cuda'

img_paths = sorted([p for p in INCOMING.iterdir()
                    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")])

print("Incoming:", len(img_paths))


notes = {
    "params": dict(
        CLASS_ID=CLASS_ID, CONF_ACCEPT=CONF_ACCEPT,
        MIN_AREA_FRAC=MIN_AREA_FRAC, MAX_AREA_FRAC=MAX_AREA_FRAC,
        BATCH=BATCH, IMGSZ=IMGSZ, DEVICE=DEVICE, model=str(MODEL_PATH)
    ),
    "results": {}
}

accepted = 0
review = 0

for start in range(0, len(img_paths), BATCH):
    batch_paths = img_paths[start:start+BATCH]
    batch_strs = [str(p) for p in batch_paths]

    # stream=True reduces memory spikes
    try:
        batch_results = list(model.predict(
            source=batch_strs,
            conf=PRED_CONF,
            imgsz=IMGSZ,
            device=DEVICE,
            stream=True,
            verbose=False
        ))
    except Exception as e:
        # if batch inference fails, mark all as review and keep going
        for p in batch_paths:
            notes["results"][p.name] = {"status": "review", "reason": "batch_predict_failed", "error": str(e)}
            review += 1
        continue

    for img_path, res in zip(batch_paths, batch_results):
        fname = img_path.name
        try:
            img = res.orig_img
            if img is None:
                notes["results"][fname] = {"status": "review", "reason": "orig_img_missing"}
                review += 1
                continue

            h, w = img.shape[:2]
            idx, conf_val = choose_best_instance(res)

            if idx is None:
                cv2.imwrite(str(REVIEW_IMG / fname), img)
                notes["results"][fname] = {"status": "review", "reason": "no_mask"}
                review += 1
                continue

            poly_px  = res.masks.xy[idx].astype(np.float32)   
            poly_xyn = res.masks.xyn[idx].astype(np.float32)
            area_frac = contour_area(poly_px) / float(w * h)

            if conf_val is not None and conf_val < CONF_ACCEPT:
                cv2.imwrite(str(REVIEW_IMG / fname), img)
                notes["results"][fname] = {"status": "review", "reason": "low_conf", "conf": conf_val, "area_frac": area_frac}
                review += 1
                continue

            if not (MIN_AREA_FRAC <= area_frac <= MAX_AREA_FRAC):
                cv2.imwrite(str(REVIEW_IMG / fname), img)
                notes["results"][fname] = {"status": "review", "reason": "area_out_of_range", "conf": conf_val, "area_frac": area_frac}
                review += 1
                continue
            
            cv2.imwrite(str(ACCEPTED_IMG / fname), img)
            flat = " ".join(f"{v:.6f}" for v in np.clip(poly_xyn, 0.0, 1.0).reshape(-1))
            (ACCEPTED_LBL / f"{img_path.stem}.txt").write_text(f"{CLASS_ID} {flat}\n")

            notes["results"][fname] = {"status": "accepted", "conf": conf_val, "area_frac": area_frac, "n_points": int(poly_xyn.shape[0])}
            accepted += 1

        except Exception as e:
            if res.orig_img is not None:
                cv2.imwrite(str(REVIEW_IMG / fname), res.orig_img)
            notes["results"][fname] = {"status": "review", "reason": "exception", "error": str(e)}
            review += 1


notes_path = META / "auto_label_notes.json"
notes_path.write_text(json.dumps(notes, indent=2))

print("✅ done")
print("Accepted:", accepted)
print("Review:", review)
print("Notes:", notes_path)

Get reasons for image mask rejections

In [ ]:
notes_path = Path("..").resolve() / "data" / "meta" / "auto_label_notes.json"
notes = json.loads(notes_path.read_text())

reasons = Counter()
for r in notes["results"].values():
    if r["status"] == "review":
        reasons[r.get("reason", "unknown")] += 1

print("Total processed:", len(notes["results"]))
print("Rejected reason counts:")
for k, v in reasons.most_common():
    print(f"  {k}: {v}")

Loop through auto-accepted images for final manual review

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil

import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- CONFIG ----------
BASE = REPO / "data"

SRC_IMG = BASE / "auto_accepted" / "images"   # change if needed
SRC_LBL = BASE / "auto_accepted" / "labels"

DST_IMG = BASE / "raw" / "images"
DST_LBL = BASE / "raw" / "labels"

REJ_DIR = BASE / "rejected_auto"
REJ_IMG = REJ_DIR / "images"
REJ_LBL = REJ_DIR / "labels"

SKIP_DIR = BASE / "skipped_auto"
SKIP_IMG = SKIP_DIR / "images"
SKIP_LBL = SKIP_DIR / "labels"

WACK_DIR = BASE / "wack_ass_imgs"
FIXABLE_DIR = BASE / "imgs_that_need_help"

for d in [DST_IMG, DST_LBL, REJ_IMG, REJ_LBL, SKIP_IMG, SKIP_LBL, WACK_DIR, FIXABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WARP_OUT_W = 512
SHOW_ORIGINAL = True  # set False for faster reviewing
IMGSZ = 1024

# ---------- HELPERS ----------
def read_yolo_seg_poly(label_path: Path):
    """
    Reads a YOLO-seg label line: class x1 y1 x2 y2 ...
    Returns Nx2 float32 coords (usually normalized 0..1).
    """
    txt = label_path.read_text().strip() if label_path.exists() else ""
    if not txt:
        return None
    parts = txt.split()
    if len(parts) < 3:
        return None
    coords = np.array(list(map(float, parts[1:])), dtype=np.float32)
    if coords.size < 6 or coords.size % 2 != 0:
        return None
    return coords.reshape(-1, 2)

def pts_to_pixels(pts, w, h):
    """
    Accepts either normalized (0..1-ish) or pixel coords.
    If max <= 1.5 -> treat as normalized.
    """
    pts = np.asarray(pts, dtype=np.float32)
    m = float(np.nanmax(pts)) if pts.size else 0.0
    if m <= 1.5:
        return pts * np.array([[w, h]], dtype=np.float32)
    return pts

def order_points(pts4):
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts4.sum(axis=1)
    rect[0] = pts4[np.argmin(s)]  # tl
    rect[2] = pts4[np.argmax(s)]  # br
    d = np.diff(pts4, axis=1).reshape(-1)
    rect[1] = pts4[np.argmin(d)]  # tr
    rect[3] = pts4[np.argmax(d)]  # bl
    return rect

def scaled_thickness(H, W, base=2):
    # ~10px on 3000x4000, ~2px on 640x640
    return max(base, int(round(min(H, W) / 300)))

def overlay_raw_polygon(img_bgr, pts_any, color=(0, 255, 0), thickness=None):
    """
    Draw the raw YOLO segmentation polygon onto the image.
    Works for normalized or pixel coords. Uses scaled thickness for large images.
    """
    H, W = img_bgr.shape[:2]
    if thickness is None:
        thickness = scaled_thickness(H, W)

    pts_px = pts_to_pixels(pts_any, W, H).astype(np.int32)
    pts_px = pts_px.reshape(-1, 1, 2)  # OpenCV-friendly shape

    vis = img_bgr.copy()
    cv2.polylines(vis, [pts_px], True, color, thickness, lineType=cv2.LINE_AA)
    return vis

def overlay_box(img_bgr, box4, color=(0, 0, 255), thickness=None):
    """
    Draw a 4-corner box onto the image.
    """
    H, W = img_bgr.shape[:2]
    if thickness is None:
        thickness = scaled_thickness(H, W)

    pts = box4.astype(np.int32).reshape(-1, 1, 2)
    vis = img_bgr.copy()
    cv2.polylines(vis, [pts], True, color, thickness, lineType=cv2.LINE_AA)
    return vis

def overlay_mask_alpha(img_bgr, mask_u8, alpha=0.35):
    """
    Overlays a binary mask (uint8 0..255) on the image as red tint.
    """
    vis = img_bgr.copy()
    if mask_u8 is None:
        return vis
    m = (mask_u8 > 0).astype(np.uint8)
    overlay = vis.copy()
    overlay[m == 1] = (0, 0, 255)  # red in BGR
    return cv2.addWeighted(overlay, alpha, vis, 1 - alpha, 0)

def robust_corners_from_polygon(img_bgr, pts_any,
                                close_k_frac=0.01,
                                trim_q=0.01,
                                eps_frac=0.02):
    """
    Robustly derive a 4-corner box from an irregular polygon:
      - rasterize -> morphological close
      - keep largest CC
      - contour points -> quantile trim to kill spikes
      - convex hull -> approxPolyDP to try for 4 corners
      - fallback: minAreaRect

    Returns:
      box4 (float32 4x2, ordered tl,tr,br,bl),
      mask_clean (uint8),
      approx_n (int),
      method (str)
    """
    H, W = img_bgr.shape[:2]
    pts_px = pts_to_pixels(pts_any, W, H).astype(np.float32)

    # --- rasterize polygon to mask ---
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(mask, [pts_px.astype(np.int32)], 255)

    # --- cleanup ---
    k = max(3, int(round(close_k_frac * min(H, W))))
    if k % 2 == 0:
        k += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # keep largest connected component
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest).astype(np.uint8) * 255

    # find contour
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None, mask, None, "none"

    cnt = max(cnts, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2).astype(np.float32)
    if pts.shape[0] < 10:
        return None, mask, pts.shape[0], "too_few"

    # --- outlier trim ---
    xs, ys = pts[:, 0], pts[:, 1]
    x_lo, x_hi = np.quantile(xs, [trim_q, 1.0 - trim_q])
    y_lo, y_hi = np.quantile(ys, [trim_q, 1.0 - trim_q])
    keep = (xs >= x_lo) & (xs <= x_hi) & (ys >= y_lo) & (ys <= y_hi)
    pts_in = pts[keep]
    if pts_in.shape[0] < 10:
        pts_in = pts  # fallback if trimming nukes points

    # convex hull -> simplify
    hull = cv2.convexHull(pts_in.astype(np.float32)).reshape(-1, 2).astype(np.float32)
    eps = eps_frac * cv2.arcLength(hull.astype(np.float32), True)
    approx = cv2.approxPolyDP(hull.astype(np.float32), eps, True)
    approx_n = len(approx)

    if approx_n == 4:
        box = approx.reshape(4, 2).astype(np.float32)
        method = "approx4"
    else:
        rect = cv2.minAreaRect(hull.astype(np.float32))
        box = cv2.boxPoints(rect).astype(np.float32)
        method = "minAreaRect"

    box = order_points(box)
    return box, mask, approx_n, method

def warp_from_box(img_bgr, box4, out_w=512):
    (tl, tr, br, bl) = box4
    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)

    W = int(max(widthA, widthB))
    H = int(max(heightA, heightB))

    scale = out_w / max(W, 1)
    W = int(out_w)
    H = int(max(1, round(H * scale)))

    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(box4.astype(np.float32), dst)
    warped = cv2.warpPerspective(img_bgr, M, (W, H), flags=cv2.INTER_LINEAR)
    return warped

def move_pair(img_path: Path, lbl_path: Path, dst_img_dir: Path, dst_lbl_dir: Path):
    shutil.move(str(img_path), str(dst_img_dir / img_path.name))
    if lbl_path is not None:
        shutil.move(str(lbl_path), str(dst_lbl_dir / lbl_path.name))

# ---------- STATE ----------
imgs = sorted(SRC_IMG.glob("*.jpg"))
state = {"i": 0}

out = widgets.Output()
status = widgets.HTML()

btn_accept = widgets.Button(description="✅ Accept", button_style="success")
btn_reject = widgets.Button(description="❌ Reject", button_style="danger")
btn_skip   = widgets.Button(description="⏭️ Skip", button_style="")
btn_quit   = widgets.Button(description="🛑 Quit", button_style="warning")
btn_exile  = widgets.Button(description ="❌❌❌ Send to the Shadow Realm", button_style="danger")
btn_needs_help  = widgets.Button(description ="Fixable!", button_style="warning")

buttons = widgets.HBox([btn_accept, btn_reject, btn_skip, btn_exile, btn_quit, btn_needs_help])

def render_current():
    with out:
        clear_output(wait=True)

        if state["i"] >= len(imgs):
            print("Done! No more images to review.")
            return

        img_path = imgs[state["i"]]
        lbl_path = SRC_LBL / f"{img_path.stem}.txt"

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print("Could not read image:", img_path)
            return
        img_bgr = cv2.resize(img_bgr, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)

        pts = read_yolo_seg_poly(lbl_path)
        if pts is None:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(6, 6))
            plt.imshow(img_rgb)
            plt.title(f"{img_path.name}\n(NO / BAD LABEL)")
            plt.axis("off")
            plt.show()
            status.value = f"<b>Reviewing:</b> {state['i']+1} / {len(imgs)} &nbsp;&nbsp; <code>{img_path.name}</code>"
            return

        # --- derive corners & warp ---
        box, mask_clean, approx_n, method = robust_corners_from_polygon(
            img_bgr, pts, trim_q=0.01, eps_frac=0.02
        )

        if box is None:
            # show raw + mask only
            vis_poly = overlay_raw_polygon(img_bgr, pts)
            vis_mask = overlay_mask_alpha(img_bgr, mask_clean, alpha=0.35)
            plt.figure(figsize=(18, 6))
            plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(vis_poly, cv2.COLOR_BGR2RGB)); plt.title("Raw polygon (green)"); plt.axis("off")
            plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(vis_mask, cv2.COLOR_BGR2RGB)); plt.title("Mask overlay (red)"); plt.axis("off")
            plt.tight_layout(); plt.show()
            status.value = f"<b>Reviewing:</b> {state['i']+1} / {len(imgs)} &nbsp;&nbsp; <code>{img_path.name}</code>"
            return
        
        warped_bgr = warp_from_box(img_bgr, box, out_w=WARP_OUT_W)

        # --- visuals (scaled thickness + correct OpenCV shapes) ---
        vis_box  = overlay_box(img_bgr, box)                 # red
        vis_poly = overlay_raw_polygon(img_bgr, pts)         # green
        vis_mask = overlay_mask_alpha(img_bgr, mask_clean)   # red tint mask

        # --- sanity prints (optional; comment out later) ---
        H, W = img_bgr.shape[:2]
        print("Image:", W, "x", H)
        print("Box range x:", float(box[:,0].min()), float(box[:,0].max()),
              " y:", float(box[:,1].min()), float(box[:,1].max()))
        print("Poly range (raw) max:", float(np.max(pts)))

        # --- plot (4 panels makes debugging way easier) ---
        plt.figure(figsize=(24, 6))
        plt.subplot(1, 3, 1)
        plt.imshow(cv2.cvtColor(vis_box, cv2.COLOR_BGR2RGB))
        plt.title(f"Box (red) via {method}, approx_n={approx_n}")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2RGB))
        plt.title("Warped")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(cv2.cvtColor(vis_poly, cv2.COLOR_BGR2RGB))
        plt.title("Raw polygon (green)")
        plt.axis("off")

        # plt.subplot(1, 4, 4)
        # plt.imshow(cv2.cvtColor(vis_mask, cv2.COLOR_BGR2RGB))
        # plt.title("Mask overlay (red)")
        # plt.axis("off")

        plt.suptitle(img_path.name, y=1.02)
        plt.tight_layout()
        plt.show()

        status.value = f"<b>Reviewing:</b> {state['i']+1} / {len(imgs)} &nbsp;&nbsp; <code>{img_path.name}</code>"

def act(kind: str):
    if state["i"] >= len(imgs):
        return

    img_path = imgs[state["i"]]
    lbl_path = SRC_LBL / f"{img_path.stem}.txt"

    if kind == "accept":
        move_pair(img_path, lbl_path, DST_IMG, DST_LBL)
    elif kind == "reject":
        move_pair(img_path, lbl_path, REJ_IMG, REJ_LBL)
    elif kind == "skip":
        move_pair(img_path, lbl_path, SKIP_IMG, SKIP_LBL)
    elif kind == "banish":
        move_pair(img_path, None, WACK_DIR, None)
    elif kind == "send_for_edits":
        move_pair(img_path, None,  FIXABLE_DIR,None)
    elif kind == "quit":
        with out:
            clear_output(wait=True)
            print("Stopped.")
        return

    state["i"] += 1
    render_current()

btn_accept.on_click(lambda _: act("accept"))
btn_reject.on_click(lambda _: act("reject"))
btn_skip.on_click(lambda _: act("skip"))
btn_quit.on_click(lambda _: act("quit"))
btn_exile.on_click(lambda _: act("banish"))
btn_needs_help.on_click(lambda _: act("send_for_edits"))

display(status, buttons, out)
render_current()

# Notes from manual review

- Notes on Data 1:

- Notes on Data 2:
    - v9 does poorly on empty trays (only dirt), cut trays, images taken with the focal tray in landscape (all of Data 1 was portrait)
    - Some images have the focal tray edges out of frame
    - images 457X have new tray type
    - images ~4400 - 4490 are predominately landscape
    - Images to remove (?):
        - No grid: 4073-79, 4136-39 
            - Put in a folder to review in failure mode discussion along with images containing excessive foliage
        - Landscape: 4230-4249, 4347, 4381-4457
            - Put in folder to rotate to portrait for future labelling

Look at masks & confidence predictions on an image-by-image basis in a folder specified by 'IMG_DIR'

In [ ]:
# ---------------- CONFIG ----------------
BASE = REPO / 'data'
IMG_DIR   = BASE / "raw" / "images"
LBL_DIR   = BASE / "raw" / "labels"   # set to None to always predict
MODEL_PATH = REPO / "models" / "trayseg_v13.pt"
DEMO_DIR  = BASE / "demo_picks"

CONF_THRES = 0.25
IOU_THRES  = 0.7
WARP_OUT_W = 512
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

DEMO_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- WARPING HELPERS (from review notebook) ----------------

def _read_yolo_seg_poly(label_path: Path):
    txt = label_path.read_text().strip() if label_path.exists() else ""
    if not txt:
        return None
    parts = txt.split()
    if len(parts) < 3:
        return None
    coords = np.array(list(map(float, parts[1:])), dtype=np.float32)
    if coords.size < 6 or coords.size % 2 != 0:
        return None
    return coords.reshape(-1, 2)

def _pts_to_pixels(pts, w, h):
    pts = np.asarray(pts, dtype=np.float32)
    m = float(np.nanmax(pts)) if pts.size else 0.0
    if m <= 1.5:
        return pts * np.array([[w, h]], dtype=np.float32)
    return pts

def _order_points(pts4):
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts4.sum(axis=1)
    rect[0] = pts4[np.argmin(s)]   # tl
    rect[2] = pts4[np.argmax(s)]   # br
    d = np.diff(pts4, axis=1).reshape(-1)
    rect[1] = pts4[np.argmin(d)]   # tr
    rect[3] = pts4[np.argmax(d)]   # bl
    return rect

def _robust_corners_from_polygon(img_bgr, pts_any,
                                  close_k_frac=0.01, trim_q=0.01, eps_frac=0.02):
    H, W = img_bgr.shape[:2]
    pts_px = _pts_to_pixels(pts_any, W, H).astype(np.float32)

    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(mask, [pts_px.astype(np.int32)], 255)

    k = max(3, int(round(close_k_frac * min(H, W))))
    if k % 2 == 0:
        k += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest).astype(np.uint8) * 255

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None, mask, None, "none"

    cnt = max(cnts, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2).astype(np.float32)
    if pts.shape[0] < 10:
        return None, mask, pts.shape[0], "too_few"

    xs, ys = pts[:, 0], pts[:, 1]
    x_lo, x_hi = np.quantile(xs, [trim_q, 1.0 - trim_q])
    y_lo, y_hi = np.quantile(ys, [trim_q, 1.0 - trim_q])
    keep = (xs >= x_lo) & (xs <= x_hi) & (ys >= y_lo) & (ys <= y_hi)
    pts_in = pts[keep]
    if pts_in.shape[0] < 10:
        pts_in = pts

    hull = cv2.convexHull(pts_in.astype(np.float32)).reshape(-1, 2).astype(np.float32)
    eps  = eps_frac * cv2.arcLength(hull.astype(np.float32), True)
    approx = cv2.approxPolyDP(hull.astype(np.float32), eps, True)
    approx_n = len(approx)

    if approx_n == 4:
        box    = approx.reshape(4, 2).astype(np.float32)
        method = "approx4"
    else:
        rect   = cv2.minAreaRect(hull.astype(np.float32))
        box    = cv2.boxPoints(rect).astype(np.float32)
        method = "minAreaRect"

    box = _order_points(box)
    return box, mask, approx_n, method

def _warp_from_box(img_bgr, box4, out_w=512):
    (tl, tr, br, bl) = box4
    widthA  = np.linalg.norm(br - bl)
    widthB  = np.linalg.norm(tr - tl)
    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)

    W = int(max(widthA, widthB))
    H = int(max(heightA, heightB))

    scale = out_w / max(W, 1)
    W = int(out_w)
    H = int(max(1, round(H * scale)))

    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M   = cv2.getPerspectiveTransform(box4.astype(np.float32), dst)
    return cv2.warpPerspective(img_bgr, M, (W, H), flags=cv2.INTER_LINEAR)

def _get_warped(img_bgr, pts_any):
    """Returns warped BGR image or None if it can't be computed."""
    if pts_any is None:
        return None
    box, _, _, _ = _robust_corners_from_polygon(img_bgr, pts_any)
    if box is None:
        return None
    return _warp_from_box(img_bgr, box, out_w=WARP_OUT_W)

# ---------------- LOAD MODEL + FILE LIST ----------------

model = YOLO(str(MODEL_PATH))

img_paths = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in IMG_EXTS])
assert len(img_paths) > 0, f"No images found in {IMG_DIR}"

# ---------------- WIDGETS ----------------
idx_slider = widgets.IntSlider(value=0, min=0, max=len(img_paths)-1, step=1, description="Index")
dropdown   = widgets.Dropdown(options=[(p.name, i) for i, p in enumerate(img_paths)],
                               value=0, description="File",
                               layout=widgets.Layout(width="60%"))

btn_prev   = widgets.Button(description="◀ Prev",    layout=widgets.Layout(width="120px"))
btn_next   = widgets.Button(description="Next ▶",    layout=widgets.Layout(width="120px"))
btn_rerun  = widgets.Button(description="Re-run",    layout=widgets.Layout(width="120px"))
btn_demo   = widgets.Button(description="⭐ Save to Demo", button_style="success",
                             layout=widgets.Layout(width="160px"))

conf_box   = widgets.FloatSlider(value=CONF_THRES, min=0.0, max=1.0, step=0.01, description="conf")
iou_box    = widgets.FloatSlider(value=IOU_THRES,  min=0.0, max=1.0, step=0.01, description="iou")

demo_status = widgets.HTML(value="")
out         = widgets.Output()

# cache: key -> (overlay_bgr, warped_bgr_or_None, summary_lines, source_tag)
_cache = {}

# keep track of the warped image for the current frame so Save works
_current_warped = {"bgr": None, "name": None}

# ---------------- THICKNESS HELPER ----------------

def auto_thickness(h, w, base=1080, min_t=2, max_t=12, scale=1.0):
    ref = max(h, w)
    t   = int(round(scale * (ref / base) * 3))
    return int(np.clip(t, min_t, max_t))

# ---------------- PREDICT / LABEL LOGIC ----------------

def _polygons_from_label(lbl_path: Path, img_h, img_w):
    """
    Parse a YOLO-seg .txt file.  Returns list of (class_id, pts_px Nx2 int32).
    """
    if not lbl_path.exists():
        return None   # file missing -> fall back to prediction
    lines = lbl_path.read_text().strip().splitlines()
    if not lines:
        return []     # file exists but empty -> no detections
    result = []
    for line in lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        cls_id = int(parts[0])
        coords = np.array(list(map(float, parts[1:])), dtype=np.float32)
        if coords.size % 2 != 0:
            continue
        pts_norm = coords.reshape(-1, 2)
        pts_px   = (pts_norm * np.array([[img_w, img_h]])).astype(np.int32)
        result.append((cls_id, pts_px))
    return result


def _predict_and_draw(img_path: Path, conf: float, iou: float, thickness: int = None):
    bgr = cv2.imread(str(img_path))
    bgr = cv2.resize(bgr, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
    if bgr is None:
        overlay = np.zeros((300, 600, 3), dtype=np.uint8)
        return overlay, None, [f"ERROR: could not read {img_path.name}"], "error"

    h, w = bgr.shape[:2]
    if thickness is None:
        thickness = auto_thickness(h, w)

    key = (str(img_path), float(conf), float(iou), int(thickness))
    if key in _cache:
        return _cache[key]

    overlay  = bgr.copy()
    summary  = []
    pts_for_warp = None   # first detection's polygon for warping

    # ---- Try label file first ----
    lbl_path = LBL_DIR / f"{img_path.stem}.txt" if LBL_DIR is not None else None
    polys    = _polygons_from_label(lbl_path, h, w) if lbl_path is not None else None

    if polys is not None:
        # --- draw from labels ---
        source_tag = "label"
        summary.append(f"Source: label file  ({len(polys)} detection(s))")
        for cls_id, pts_px in polys:
            pts_cv = pts_px.reshape(-1, 1, 2)
            cv2.polylines(overlay, [pts_cv], isClosed=True,
                          color=(0, 255, 0), thickness=thickness)
            if pts_for_warp is None:
                # store normalized for warping helper
                pts_for_warp = pts_px.astype(np.float32) / np.array([[w, h]])
    else:
        # --- fall back to model prediction ---
        source_tag = "model"
        summary.append(f"Source: model prediction")

        results = model.predict(source=bgr, conf=conf, iou=iou, verbose=False)
        r       = results[0]
        names   = r.names

        if r.boxes is not None and len(r.boxes) > 0:
            cls_ids = r.boxes.cls.cpu().numpy().astype(int)
            confs   = r.boxes.conf.cpu().numpy()
            order   = np.argsort(-confs)
            summary.append(f"Detections: {len(confs)}")
            for j in order[:10]:
                summary.append(f"  {names.get(cls_ids[j], str(cls_ids[j]))}: {confs[j]:.3f}")
            if len(confs) > 10:
                summary.append(f"  ... (+{len(confs)-10} more)")
        else:
            summary.append("Detections: 0")

        drew_any = False
        if r.masks is not None and getattr(r.masks, "xy", None) is not None:
            for poly in r.masks.xy:
                if poly is None or len(poly) < 3:
                    continue
                pts_cv = poly.astype(np.int32).reshape(-1, 1, 2)
                cv2.polylines(overlay, [pts_cv], isClosed=True,
                              color=(0, 255, 0), thickness=thickness)
                drew_any = True
                if pts_for_warp is None:
                    pts_for_warp = poly.astype(np.float32) / np.array([[w, h]])

        if not drew_any and r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy().astype(int)
            for (x1, y1, x2, y2) in xyxy:
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), thickness)

    # ---- compute warp ----
    warped = _get_warped(bgr, pts_for_warp)

    _cache[key] = (overlay, warped, summary, source_tag)
    return overlay, warped, summary, source_tag


# ---------------- RENDER ----------------

def render(i: int):
    img_path = img_paths[i]
    conf = float(conf_box.value)
    iou  = float(iou_box.value)

    overlay_bgr, warped_bgr, summary, source_tag = _predict_and_draw(img_path, conf, iou)

    # stash for demo-save
    _current_warped["bgr"]  = warped_bgr
    _current_warped["name"] = img_path.name

    demo_status.value = ""   # clear previous save message

    with out:
        clear_output(wait=True)

        overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

        if warped_bgr is not None:
            warped_rgb = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2RGB)
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            axes[0].imshow(overlay_rgb);  axes[0].axis("off")
            axes[0].set_title(f"{img_path.name}  [{source_tag}]  ({i+1}/{len(img_paths)})")
            axes[1].imshow(warped_rgb);   axes[1].axis("off")
            axes[1].set_title("Warped / cropped")
        else:
            fig, ax = plt.subplots(1, 1, figsize=(10, 7))
            ax.imshow(overlay_rgb);  ax.axis("off")
            ax.set_title(f"{img_path.name}  [{source_tag}]  ({i+1}/{len(img_paths)})")

        plt.tight_layout()
        plt.show()
        print("\n".join(summary))


# ---------------- CALLBACKS ----------------

def sync_from_slider(change=None):
    dropdown.value = idx_slider.value
    render(idx_slider.value)

def sync_from_dropdown(change=None):
    idx_slider.value = dropdown.value
    render(dropdown.value)

def on_prev(_):
    idx_slider.value = max(0, idx_slider.value - 1)

def on_next(_):
    idx_slider.value = min(len(img_paths) - 1, idx_slider.value + 1)

def on_rerun(_):
    img_path = img_paths[idx_slider.value]
    conf = float(conf_box.value)
    iou  = float(iou_box.value)
    for k in list(_cache.keys()):
        if k[0] == str(img_path) and k[1] == conf and k[2] == iou:
            _cache.pop(k, None)
    render(idx_slider.value)

def on_save_demo(_):
    warped = _current_warped["bgr"]
    name   = _current_warped["name"]
    if warped is None or name is None:
        demo_status.value = "<span style='color:orange'>⚠️ No warped image to save.</span>"
        return
    dst = DEMO_DIR / name
    cv2.imwrite(str(dst), warped)
    demo_status.value = f"<span style='color:green'>✅ Saved → <code>{dst}</code></span>"

def on_thresh_change(_):
    render(idx_slider.value)

idx_slider.observe(sync_from_slider, names="value")
dropdown.observe(sync_from_dropdown,  names="value")
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_rerun.on_click(on_rerun)
btn_demo.on_click(on_save_demo)
conf_box.observe(on_thresh_change, names="value")
iou_box.observe(on_thresh_change,  names="value")

# ---------------- LAYOUT ----------------

controls   = widgets.HBox([btn_prev, btn_next, btn_rerun, btn_demo, demo_status])
selectors  = widgets.HBox([idx_slider, dropdown])
thresholds = widgets.HBox([conf_box, iou_box])

display(widgets.VBox([selectors, thresholds, controls, out]))
render(0)

List of images to include in demo: 
Sparse foliage, trays close together: 
4044, 4059, 4090, 4180, 4250

Dense foliage, trays close together:
4056, 4175, 4176, 4220, 4252

Sparse foliage, trays far apart:
4072, 4178, 4199, 

Dense foliage, trays far apart: 
4174, 4211, 

Double Detections w high confidence: 4213, 4224

In [ ]:
BASE = REPO / "data"
SRC_DIR = BASE / "new_images_2_16_jpg"
DST_DIR = BASE / "demo_images_2_17"

DST_DIR.mkdir(exist_ok=True)

# ---- IDs you listed ----
ids = [
    4044, 4059, 4090, 4180, 4250,      # sparse / close
    4056, 4175, 4176, 4220, 4252,      # dense / close
    4072, 4178, 4199,                  # sparse / far
    4174, 4211,                        # dense / far
    4213, 4224                         # double detections
]

# Remove duplicates just in case
ids = sorted(set(ids))

counter = 1

for img_id in ids:
    matches = list(SRC_DIR.glob(f"*{img_id}*.jpg"))

    if not matches:
        print(f"⚠ No match for {img_id}")
        continue

    src_path = matches[0]  # assume one match
    dst_name = f"IMG_{counter:03d}.jpg"
    dst_path = DST_DIR / dst_name

    shutil.copy2(src_path, dst_path)

    print(f"{src_path.name} → {dst_name}")

    counter += 1

print("\nDone.")

# Function for per-image prediction

In [ ]:
def predict_and_crop(
    img: np.ndarray | str | Path,
    model,
    out_w: int = 512,
    conf: float = 0.25,
    iou: float = 0.7,
    imgsz: int = 1024,
) -> np.ndarray | None:
    """
    Takes a raw image (BGR array, path, or Path), runs the model,
    and returns a perspective-corrected crop of the detected tray.

    Returns the warped BGR image, or None if no detection.
    """
    # ---- load if path ----
    if isinstance(img, (str, Path)):
        img = cv2.imread(str(img))
    if img is None:
        return None

    orig_h, orig_w = img.shape[:2]

    # ---- resize to inference size ----
    img_resized = cv2.resize(img, (imgsz, imgsz), interpolation=cv2.INTER_LINEAR)

    # ---- predict ----
    results = model.predict(source=img_resized, conf=conf, iou=iou,
                            imgsz=imgsz, verbose=False)
    r = results[0]

    # ---- get best polygon ----
    pts = None
    if r.masks is not None and getattr(r.masks, "xy", None) is not None:
        polys = [p for p in r.masks.xy if p is not None and len(p) >= 3]
        if polys:
            if r.boxes is not None and len(r.boxes.conf) == len(polys):
                best = int(np.argmax(r.boxes.conf.cpu().numpy()))
            else:
                best = int(np.argmax([cv2.contourArea(p.astype(np.float32)) for p in polys]))
            pts = polys[best].astype(np.float32) / np.array([[imgsz, imgsz]])

    if pts is None:
        return None

    # ---- warp on resized image ----
    box, _, _, _ = _robust_corners_from_polygon(img_resized, pts)
    if box is None:
        return None

    return _warp_from_box(img_resized, box, out_w=out_w)

In [ ]:
model = YOLO(str(REPO / "models" / "trayseg_v18_1024.pt"))

# test on a single image
img_path = next(IMG_DIR.iterdir())  # grab any image
result = predict_and_crop(img_path, model, out_w=512, imgsz=1024)

if result is None:
    print("No detection")
else:
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(img_path.name)
    plt.show()